# Data Exploration

Quick notebook for reviewing dataset counts, label distributions, and logged teacher cost.

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from pathlib import Path

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
LOG_PATH = ROOT / "data" / "logs" / "api_costs.jsonl"

def read_json(path: Path) -> dict:
    return json.loads(path.read_text(encoding="utf-8"))

def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8").splitlines()
    return [json.loads(line) for line in lines if line.strip()]

In [ ]:
manifest = read_json(RAW_DIR / "manifest.json")
summary = read_json(PROCESSED_DIR / "label_distribution_summary.json")
manifest

In [ ]:
summary

In [ ]:
cost_rows = read_jsonl(LOG_PATH)
cost_by_dataset = defaultdict(float)
tokens_by_dataset = defaultdict(int)
for row in cost_rows:
    dataset = row.get("dataset", "unknown")
    cost_by_dataset[dataset] += float(row.get("cost_usd", 0.0))
    tokens_by_dataset[dataset] += int(row.get("total_tokens", 0))

{
    "total_cost_usd": round(sum(cost_by_dataset.values()), 6),
    "cost_by_dataset": {key: round(value, 6) for key, value in sorted(cost_by_dataset.items())},
    "tokens_by_dataset": dict(sorted(tokens_by_dataset.items())),
    "request_count": len(cost_rows),
}

In [ ]:
pair_rows = read_jsonl(RAW_DIR / "pairs.jsonl")
preference_rows = read_jsonl(RAW_DIR / "preferences.jsonl")
contradiction_rows = read_jsonl(RAW_DIR / "contradictions.jsonl")

{
    "pairs_preview": pair_rows[:2],
    "preferences_preview": preference_rows[:2],
    "contradictions_preview": contradiction_rows[:2],
}

## Label Distribution Charts

In [ ]:
from IPython.display import Image, display

chart_path = PROCESSED_DIR / "label_distribution.png"
if chart_path.exists():
    display(Image(filename=str(chart_path)))
else:
    print("No chart found. Run data generation with matplotlib installed to produce one.")

## Dataset Statistics

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Synthetic Dataset Label Distributions", fontsize=14, fontweight="bold")

datasets = [
    ("pairs", pair_rows, "score", "Relevance Score"),
    ("preferences", preference_rows, "preferred", "Preferred Document"),
    ("contradictions", contradiction_rows, "is_contradiction", "Contradiction?"),
]

for ax, (name, rows, key, title) in zip(axes, datasets, strict=False):
    if not rows:
        ax.text(0.5, 0.5, "No data", ha="center", va="center", transform=ax.transAxes)
        ax.set_title(name)
        continue

    values = [str(r[key]) for r in rows]
    unique, counts = [], []
    seen = {}
    for v in values:
        if v not in seen:
            seen[v] = 0
            unique.append(v)
        seen[v] += 1
    counts = [seen[v] for v in unique]

    colors = ["#4C72B0", "#55A868", "#C44E52", "#8172B2", "#CCB974"]
    bars = ax.bar(range(len(unique)), counts, color=colors[:len(unique)])
    ax.set_xticks(range(len(unique)))
    ax.set_xticklabels(unique, fontsize=10)
    ax.set_title(f"{title}\n(n={len(rows)})", fontsize=11)
    ax.set_ylabel("Count", fontsize=10)

    for bar, count in zip(bars, counts, strict=False):
        ax.text(bar.get_x() + bar.get_width() / 2.0, bar.get_height() + max(count * 0.01, 0.5),
                str(count), ha="center", va="bottom", fontsize=9, fontweight="bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

plt.tight_layout()
plt.show()

## Parquet Support (Optional)

If `pyarrow` is installed, data can also be stored in Parquet format for faster I/O.

In [ ]:
try:
    import pyarrow
    import pyarrow.parquet as pq

    parquet_dir = PROCESSED_DIR / "parquet"
    parquet_dir.mkdir(parents=True, exist_ok=True)

    datasets = [
        ("pairs", pair_rows),
        ("preferences", preference_rows),
        ("contradictions", contradiction_rows),
    ]

    for name, rows in datasets:
        if rows:
            path = parquet_dir / f"{name}.parquet"
            table = pyarrow.Table.from_pylist(rows)
            pq.write_table(table, str(path))
            print(f"Wrote {len(rows)} rows to {path}")

    print("\nParquet files written successfully.")
except ImportError:
    print("pyarrow not installed. Install with: pip install pyarrow")